# Create Klysron and Linac overlays

In [ ]:
from pytao import Tao
from pprint import pprint
import json
import os
from math import sqrt

In [ ]:
M = Tao('-init $LCLS_LATTICE/bmad/models/cu_linac/tao.init -noplot')

In [ ]:
from pytao.tao_ctypes.util import parse_bool, parse_tao_lat_ele_list

In [ ]:
SLIST = M.lat_list('*', 'ele.s', flags='-array_out -no_slaves')
LLIST = M.lat_list('*', 'ele.l', flags='-array_out -no_slaves')
NAMES = M.lat_list('*', 'ele.name', flags='-no_slaves')

for s, l, n in zip(SLIST[0:20], LLIST[0:20], NAMES[0:20]):
    print (n, l, s)

In [ ]:
# Index lookup function
ix_of =parse_tao_lat_ele_list(M.lat_ele_list('0'))
s_of = {}
for n,s in zip(NAMES, SLIST):
    s_of[n] = s

In [ ]:
'K21_1B' in NAMES

In [ ]:
s_of['K21_4B']

In [ ]:
CAVS = [name for name in NAMES if name.startswith('K') and ('#' not in name)]
KLYS = {}
for cav in CAVS:
    name, section = cav[0:-1], cav[-1:]
    if name not in KLYS:
        KLYS[name] = []
    KLYS[name].append(section)
    
def eles_in(sector, station):
    name = f'K{sector}_{station}'
    sections = KLYS[name]
    return [name+s for s in sections]


KLYS

In [ ]:
# Get elements
eles_in(30, 8)

In [ ]:
#dat = M.cmd('python ele:gen_attribs 1@0>>1491|model')
params = M.ele_gen_attribs(1491)


In [ ]:
# Get all parameters for cavities
CAVDAT = {}
EXTRACAVS = ['L1X']
for name in CAVS+EXTRACAVS:
    ix = ix_of[name]
    #print(name, ix)
    params = M.ele_gen_attribs(ix)
    CAVDAT[name] = params
CAVDAT['K21_1B']['VOLTAGE'], CAVDAT['L1X']['VOLTAGE']

In [ ]:
{'A', 'B'} == {'B', 'A'}

In [ ]:

# Klystron power divisions when there is a missing section:
POWER_FACTOR = {
    ("A", "B", "C", "D"): (0.25, 0.25, 0.25, 0.25),
    ("B", "C", "D"): (0.5, 0.25, 0.25),
    ("A", "C", "D"): (0.5, 0.25, 0.25),
    ("A", "B", "C"): (0.25, 0.25, 0.5),
    ("A", "B", "D"): (0.25, 0.25, 0.5),
}
POWER_FACTOR[('A', 'B', 'D')]

In [ ]:
def voltage_factors(sections):
    """
    Voltages go as the sqrt of the power
    """
    s = tuple(sections)
    pfactors = POWER_FACTOR[s]
    vfactors = [sqrt(p) for p in pfactors]
    vtot = sum(vfactors)
    vfactors = [v/vtot for v in vfactors]
    
    return vfactors
voltage_factors(['A', 'B', 'D'])
        

In [ ]:
def klys_data(name, sections):
    """
    
    name: 'K21_1'
    sections = ['B', 'C', 'D']
    
    
    """
    eles = [name+s for s in sections]
    voltages = [CAVDAT[ele]['VOLTAGE'] for ele in eles]
    phi0s = [CAVDAT[ele]['PHI0'] for ele in eles]
    gradients = [CAVDAT[ele]['GRADIENT'] for ele in eles]
    lengths = [CAVDAT[ele]['L'] for ele in eles]
    
    # Error checking
    assert len(set(phi0s)) == 1, 'phi0 are not unique'
    phi0 = phi0s[0]

    vtot = sum(voltages)
    vfactors = voltage_factors(sections)
    new_gradients = [vf*vtot/ l  for l, vf in zip( lengths, vfactors) ]
    
    
    # Overlay stuff
    
   #oname = 'O_'+name
    oname = name
    
    
    lines = []
    lines.append('!----------------')
    lines.append('! Klystron')
    lines.append('! Configuration: '+''.join(sections))
    lines.append('!')
    lines.append(f'{oname}: overlay = {{')

    for ele, vf, l in zip(eles, vfactors, lengths):
        lines.append(f'    {ele}[gradient]:f*in_use*ENLD_MeV*1e6*{vf}/{l},')
 
    for ele, vf, l in zip(eles, vfactors, lengths):
        lines.append(f'    {ele}[gradient_err]:f*in_use*ENLD_MeV_err*1e6*{vf}/{l},')

    for ele in eles:
        lines.append(f'    {ele}[phi0]:phase_deg/360,')
        
    for ele in eles:
        lines.append(f'    {ele}[phi0_err]:phase_deg_err/360,')        
        
    lines[-1] = lines[-1][:-1]+'}, var = {ENLD_MeV, ENLD_MeV_err, phase_deg, phase_deg_err, f, in_use}, f=1, in_use=1'
    lines.append('\n')
    overlay = '\n'.join(lines)
    
    # Design settings
    # Note that the phase is the local phase. The overall phase will be added by another overlay
    settings = f"""
! Design settings for {name}
{oname}[ENLD_MeV] = {vtot*1e-6}
{oname}[phase_deg] = 0 
"""
    
    #return eles, gradients, new_gradients, overlay
    return {'settings':settings, 'overlay':overlay}
for k,v in KLYS.items():
    print(klys_data(k,v)['overlay'])
    print(klys_data(k,v)['settings'])

In [ ]:
CAVDAT['L1X']['VOLTAGE'], CAVDAT['L1X']['PHI0']*360

In [ ]:
# Special overlay for L1X

L1X_VOLTAGE = CAVDAT['L1X']['VOLTAGE']
L1X_PHI0    = CAVDAT['L1X']['PHI0']

L1X_KLYS = f"""
!---------------- 
! Special X-band klystron
K21_2: overlay = {{
    L1X[gradient]: f*in_use*ENLD_MeV*1e6/L1X[L],
    L1X[gradient_err]: f*in_use*ENLD_MeV_err*1e6/L1X[L],
    L1X[phi0]: phase_deg/360,
    L1X[phi0_err]: phase_deg_err/360}},  
    var = {{ ENLD_MeV, ENLD_MeV_err, phase_deg, phase_deg_err, f, in_use}}, 
    f=1, ENLD_MeV={L1X_VOLTAGE*1e-6}, phase_deg={round(L1X_PHI0*360,9)}, in_use=1

"""
print(L1X_KLYS)

In [ ]:
with open('../klystrons.bmad', 'w') as f:

    f.write(L1X_KLYS)
    
    for k,v in KLYS.items():
        f.write(klys_data(k,v)['overlay'])
        
    
        
with open('../klystron_design_settings.bmad', 'w') as f:
    for k,v in KLYS.items():
        f.write(klys_data(k,v)['settings'])        

# Linac grouping

In [ ]:
s_of['ENDL1'], s_of['ENDL2'], s_of['ENDL3']

In [ ]:
def linac_of(name):
    s = s_of[name]
    
    if s_of['BEGL1'] <= s <= s_of['ENDL1']:
        return 'L1'
    elif s_of['BEGL2'] <= s <= s_of['ENDL2']:
        return 'L2'    
    elif s_of['BEGL3'] <= s <= s_of['ENDL3']:
        return 'L3'       
    else:
        return None
    

In [ ]:
linac_of('K21_1D')

In [ ]:
# Collect elements
LINAC = {'L1':[],'L2':[],'L3':[]}
for n in CAVS:
    l = linac_of(n)
    LINAC[l].append(n)

In [ ]:
LINAC['L1'], len(LINAC['L2']), len(LINAC['L3'])

In [ ]:
# Find special feedback cavities

L2FEEDBACK = []
L2FORPHASE = []
for n in LINAC['L2']:
    if any([n.startswith(x) for x in ['K24_1', 'K24_2', 'K24_3']]):
        L2FEEDBACK.append(n)
    else:
        L2FORPHASE.append(n)
len(L2FEEDBACK), len(L2FORPHASE)

In [ ]:
L3FEEDBACK = []
L3FORPHASE = []
for n in LINAC['L3']:
    if any([n.startswith(x) for x in ['K29', 'K30']]):
        L3FEEDBACK.append(n)
    else:
        L3FORPHASE.append(n)
len(L3FEEDBACK), len(L3FORPHASE)

In [ ]:
def unique_param(eles, param):
    p = set()
    for ele in eles:
        p.add(CAVDAT[ele][param])
    assert len(p) == 1
    return list(p)[0]
L1phi0 = unique_param(LINAC['L1'], 'PHI0')
L2phi0 = unique_param(LINAC['L2'], 'PHI0')
L3phi0 = unique_param(LINAC['L3'], 'PHI0')

In [ ]:
L1KLYS = list(set([c[:-1] for c in LINAC['L1']]))
L1KLYS
list(set([c[:-1] for c in LINAC['L2']])).sort()
#list(set([c[:-1] for c in LINAC['L3']]))

In [ ]:
# Get klystrons by linac
L1KLYS = list(set([c[:-1] for c in LINAC['L1']]))
L2KLYS = sorted(list(set([c[:-1] for c in LINAC['L2']])))
L3KLYS = sorted(list(set([c[:-1] for c in LINAC['L3']])))
# Klystrons for L2, L3 feedback
L2FEEDBACKKLYS = sorted(list(set([c[:-1] for c in L2FEEDBACK])))
L3FEEDBACKKLYS = sorted(list(set([c[:-1] for c in L3FEEDBACK])))

In [ ]:

for name in L2FEEDBACKKLYS:
    print(f'{name}[phase_deg] = {round(L2phi0*360,9)}')

In [ ]:
# Write to file
with open('../linac_phase_defaults.bmad', 'w') as f:
    f.write('! Design linac phasing\n')
    f.write(f'O_L1[phase_deg] = {round(L1phi0*360,9)}\n')
    f.write(f'O_L2[phase_deg] = {round(L2phi0*360,9)}\n')
    f.write(f'O_L3[phase_deg] = {round(L3phi0*360,9)}\n')
    
    f.write('\n!------------------\n')
    f.write('\n! Feedback design phases\n')
    for name in L2FEEDBACKKLYS:
        f.write(f'{name}[phase_deg] = {round(L2phi0*360,9)}\n')
        
    for name in L3FEEDBACKKLYS:
        f.write(f'{name}[phase_deg] = {round(L3phi0*360,9)}\n')        
    

In [ ]:
# Fudge
def fudge_overlays(name, cavs):

    lines = []
    lines.append('!--------------\n')
    lines.append(f'{name}: overlay = {{\n')
    i=0 
    for n in cavs:
        i += 1
        lines.append(f'  {n}[f]:f,')
        if i == 4:
            i = 0
            lines.append('\n')   
    if lines[-1] == '\n':
        lines.pop()
        
    lines[-1] = lines[-1][:-1]+'}, var = {f}, f=1\n\n'
    return ''.join(lines)
print(fudge_overlays('O_L1_fudge', L1KLYS))
print(fudge_overlays('O_L2_fudge', L2KLYS))
print(fudge_overlays('O_L3_fudge', L3KLYS))

In [ ]:
# Overall phase overlays
with open('../linac_fudge_overlays.bmad', 'w') as f:
    f.write(fudge_overlays('O_L1_fudge', L1KLYS))
    f.write(fudge_overlays('O_L2_fudge', L2KLYS))
    f.write(fudge_overlays('O_L3_fudge', L3KLYS))

In [ ]:
# Make overlays
def phase_overlay(name, cavs):
    lines = []
    lines.append('!--------------\n')
    lines.append('! Linac phase overlay\n')
    lines.append(f'{name}: overlay = {{\n')
    i=0
    for n in cavs:
        i += 1
        lines.append(f'  {n}[phi0]:phase_deg/360,')
        if i == 4:
            i = 0
            lines.append('\n')   
    if lines[-1] == '\n':
        lines.pop()
        
    lines[-1] = lines[-1][:-1]+'}, var = {phase_deg}\n\n'
    return ''.join(lines)




print(phase_overlay('O_L1', LINAC['L1'])           )
print(phase_overlay('O_L2', L2FORPHASE)           )
print(phase_overlay('O_L3', L3FORPHASE)           )

# Subbooster Overlays

Subboosters adjust the phase over entire sectors.


In [ ]:
# Make Subbooster overlays
def sbst_overlay(sector, stations):
    
    name = f'SBST_{sector}'
    
    lines=f"""
!--------------
!subbooster {sector} overlay
{name}: overlay = {{
"""

    lines +=',\n'.join([', '.join([f'  {ele}[phi0]:phase_deg/360' for ele in eles_in(sector, station)]) for station in stations])


    lines += '}, var = {phase_deg}\n'
    lines += f'{name}[alias] = SBST:LI{sector}:1:PHAS'
    
    return lines
print(sbst_overlay('22', (1,2,3,4,5,6,7,8)))

# Write to file

In [ ]:
# Overall phase overlays
with open('../linac_phase_overlays.bmad', 'w') as f:
    f.write( phase_overlay('O_L1', LINAC['L1']) )
    f.write(phase_overlay('O_L2', L2FORPHASE)     )
    f.write(phase_overlay('O_L3', L3FORPHASE)     )

In [ ]:
!cat ../linac_phase_overlays.bmad

In [ ]:
SUBBOOSTER_STATIONS = {
    21:(3,4,5,6,7,8), #21-1 (L1S) and 21-2 (L1X) dont use the subbooster.
    22:(1,2,3,4,5,6,7,8),
    23:(1,2,3,4,5,6,7,8),
    24:(4,5,6), #1,2,3 are feedback stations, no sbst.  7 is a tcav, 8 doesn't exist.
    25:(1,2,3,4,5,6,7,8),
    26:(1,2,3,4,5,6,7,8),
    27:(1,2,3,4,5,6,7,8),
    28:(1,2,3,4,5,6,7,8),
    29:(1,2,3,4,5,6,7,8),
    30:(1,2,3,4,5,6,7,8)  
}

In [ ]:
list(SUBBOOSTER_STATIONS)

In [ ]:
# Subbooster overlays
with open('../sbst_phase_overlays.bmad', 'w') as f:
    for sector in SUBBOOSTER_STATIONS:
        out = sbst_overlay(sector, SUBBOOSTER_STATIONS[sector])
        print(out)
        f.write(out)
    